In [5]:
"""
DigiCow Farmer Training Adoption Challenge
==========================================
Solution ML haute performance
Modèles: HistGradientBoosting + RandomForest + Stacking
Optimisé pour: Log Loss (75%) + ROC-AUC (25%) + F1-Score

INSTRUCTIONS D'UTILISATION:
  python solution_digicow.py

  Si vous avez LightGBM/XGBoost:
    pip install lightgbm xgboost catboost
  Puis décommentez les sections correspondantes.
"""

import pandas as pd
import numpy as np
import ast
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import log_loss, roc_auc_score, f1_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (
    HistGradientBoostingClassifier,
    RandomForestClassifier,
    GradientBoostingClassifier,
)
from sklearn.calibration import CalibratedClassifierCV

# ============================================================
# OPTIONNEL: LightGBM / XGBoost / CatBoost
# Décommentez si vous avez ces packages installés
# ============================================================
USE_LGBM    = False
USE_XGB     = False
USE_CATBOOST= False

try:
    import lightgbm as lgb
    USE_LGBM = True
    print("✓ LightGBM disponible")
except ImportError:
    print("✗ LightGBM non disponible (utilisation de HistGBM à la place)")

try:
    import xgboost as xgb
    USE_XGB = True
    print("✓ XGBoost disponible")
except ImportError:
    print("✗ XGBoost non disponible")

try:
    from catboost import CatBoostClassifier
    USE_CATBOOST = True
    print("✓ CatBoost disponible")
except ImportError:
    print("✗ CatBoost non disponible")


# ============================================================
# 1. CHARGEMENT DES DONNÉES
# ============================================================

def load_data(path='./'):
    train  = pd.read_csv(f'{path}Train.csv')
    test   = pd.read_csv(f'{path}Test.csv')
    prior  = pd.read_csv(f'{path}Prior.csv')
    sample = pd.read_csv(f'{path}SampleSubmission.csv')
    return train, test, prior, sample


# ============================================================
# 2. PARSING DES COLONNES COMPLEXES
# ============================================================

def parse_topics(topics_str):
    """Parse la liste imbriquée de sujets de formation."""
    try:
        raw = ast.literal_eval(str(topics_str))
        topics = []
        for item in raw:
            if isinstance(item, list):
                topics.extend([t for t in item if isinstance(t, str)])
            elif isinstance(item, str):
                topics.append(item)
        return [t.strip().lower() for t in topics if t.strip()]
    except Exception:
        return []

def parse_trainers(trainer_str):
    """Parse la liste de formateurs."""
    try:
        raw = ast.literal_eval(str(trainer_str))
        if isinstance(raw, list):
            return [str(t) for t in raw]
        return [str(raw)]
    except Exception:
        return [str(trainer_str)]


# ============================================================
# 3. STATISTIQUES PRIOR (Target Encoding robuste)
# ============================================================

def build_prior_stats(prior):
    """
    Calcule les taux d'adoption historiques depuis Prior.csv
    Ces statistiques servent de target encoding robuste.
    """
    print("  Calcul des stats Prior...")
    prior = prior.copy()
    prior['training_day'] = pd.to_datetime(prior['training_day'])
    prior['month']     = prior['training_day'].dt.month
    prior['dayofweek'] = prior['training_day'].dt.dayofweek
    prior['topics_parsed']  = prior['topics_list'].apply(parse_topics)
    prior['trainer_first']  = prior['trainer'].apply(
        lambda x: parse_trainers(x)[0] if parse_trainers(x) else 'unknown'
    )

    stats = {}
    TARGETS_3 = ['adopted_within_07_days', 'adopted_within_90_days', 'adopted_within_120_days']

    # --- Target encoding par colonne catégorielle ---
    for col in ['county', 'subcounty', 'ward', 'group_name', 'trainer_first',
                'gender', 'age', 'registration', 'month', 'dayofweek']:
        for target in TARGETS_3:
            key = f'{col}_{target}_rate'
            # Smoothing bayésien: (count * rate + prior_count * global_rate) / (count + prior_count)
            global_rate = prior[target].mean()
            agg = prior.groupby(col)[target].agg(['sum', 'count'])
            smooth_k = 50  # force du prior
            agg['rate'] = (agg['sum'] + smooth_k * global_rate) / (agg['count'] + smooth_k)
            stats[key] = agg['rate'].to_dict()

    # --- Target encoding par sujet de formation ---
    rows = []
    for _, row in prior.iterrows():
        for topic in row['topics_parsed']:
            rows.append({
                'topic': topic,
                'adopted_within_07_days':  row['adopted_within_07_days'],
                'adopted_within_90_days':  row['adopted_within_90_days'],
                'adopted_within_120_days': row['adopted_within_120_days'],
            })
    if rows:
        topic_df = pd.DataFrame(rows)
        global_rates = {t: prior[t].mean() for t in TARGETS_3}
        smooth_k = 30
        for target in TARGETS_3:
            agg = topic_df.groupby('topic')[target].agg(['sum', 'count'])
            gr = global_rates[target]
            agg['rate'] = (agg['sum'] + smooth_k * gr) / (agg['count'] + smooth_k)
            stats[f'topic_{target}_rate'] = agg['rate'].to_dict()

    # --- Comptages ---
    stats['farmer_prior_count'] = prior['farmer_name'].value_counts().to_dict()
    stats['group_size']         = prior['group_name'].value_counts().to_dict()
    stats['trainer_event_count']= prior['trainer_first'].value_counts().to_dict()
    stats['global_rates']       = {t: prior[t].mean() for t in TARGETS_3}

    return stats


# ============================================================
# 4. FEATURE ENGINEERING
# ============================================================

TOPIC_KEYWORDS = {
    'dairy':   ['dairy', 'milk', 'cow', 'calf', 'lactating', 'calving', 'unga', 'transition'],
    'poultry': ['poultry', 'chicken', 'layer', 'broiler', 'kienyeji', 'egg', 'hatch'],
    'crop':    ['maize', 'bean', 'seed', 'fertilizer', 'weed', 'pest', 'crop', 'asili'],
    'health':  ['health', 'vaccine', 'vaccin', 'biosecurity', 'disease', 'ppe', 'hygiene'],
    'digital': ['app', 'ndume', 'record', 'digital'],
    'feeding': ['feeding', 'feed', 'nutrition', 'tyari'],
}

def feature_engineer(df, prior_stats, is_train=True):
    df = df.copy()
    gr = prior_stats['global_rates']

    # --- Dates ---
    df['training_day'] = pd.to_datetime(df['training_day'])
    df['month']      = df['training_day'].dt.month
    df['dayofweek']  = df['training_day'].dt.dayofweek
    df['week']       = df['training_day'].dt.isocalendar().week.astype(int)
    df['quarter']    = df['training_day'].dt.quarter
    df['is_weekend'] = (df['dayofweek'] >= 5).astype(int)
    df['day_of_month'] = df['training_day'].dt.day

    # --- Encodages simples ---
    age_map = {'Below 18': 0, '18-25': 1, '26-35': 2, 'Above 35': 3}
    df['age_enc'] = df['age'].map(age_map).fillna(2).astype(int)
    df['gender_enc'] = (df['gender'] == 'Female').astype(int)

    # Registration → ordinal (USSD = feature phone, Manual = basique, App = avancé)
    reg_map = {'Ussd': 1, 'Manual': 0, 'App': 2}
    df['registration_enc'] = df['registration'].map(reg_map).fillna(0).astype(int)

    # --- Topics ---
    df['topics_parsed'] = df['topics_list'].apply(parse_topics)
    df['n_topics']       = df['topics_parsed'].apply(len)
    df['n_unique_topics']= df['topics_parsed'].apply(lambda x: len(set(x)))
    df['topics_ratio']   = df['n_unique_topics'] / (df['n_topics'] + 1)

    for cat, keywords in TOPIC_KEYWORDS.items():
        df[f'n_{cat}_topics'] = df['topics_parsed'].apply(
            lambda ts: sum(any(kw in t for kw in keywords) for t in ts)
        )
    df['n_category_types'] = sum(
        (df[f'n_{cat}_topics'] > 0).astype(int) for cat in TOPIC_KEYWORDS
    )

    # Target encoding par topic
    for short, target in [('07', 'adopted_within_07_days'),
                           ('90', 'adopted_within_90_days'),
                           ('120', 'adopted_within_120_days')]:
        rate_dict = prior_stats.get(f'topic_{target}_rate', {})
        default = gr[target]
        df[f'topic_avg_{short}_rate'] = df['topics_parsed'].apply(
            lambda ts: np.mean([rate_dict.get(t, default) for t in ts]) if ts else default
        )
        df[f'topic_max_{short}_rate'] = df['topics_parsed'].apply(
            lambda ts: np.max([rate_dict.get(t, default) for t in ts]) if ts else default
        )
        df[f'topic_min_{short}_rate'] = df['topics_parsed'].apply(
            lambda ts: np.min([rate_dict.get(t, default) for t in ts]) if ts else default
        )

    # --- Trainers ---
    df['trainers_parsed']  = df['trainer'].apply(parse_trainers)
    df['n_trainers']       = df['trainers_parsed'].apply(len)
    df['trainer_first']    = df['trainers_parsed'].apply(lambda x: x[0] if x else 'unknown')
    df['trainer_event_count'] = df['trainer_first'].map(
        prior_stats.get('trainer_event_count', {})).fillna(1)

    # --- Target encoding par variables catégorielles ---
    for col, col_key in [
        ('county', 'county'), ('subcounty', 'subcounty'), ('ward', 'ward'),
        ('group_name', 'group_name'), ('trainer_first', 'trainer_first'),
        ('gender', 'gender'), ('age', 'age'), ('registration', 'registration'),
        ('month', 'month'), ('dayofweek', 'dayofweek'),
    ]:
        for short, target in [('07', 'adopted_within_07_days'),
                               ('90', 'adopted_within_90_days'),
                               ('120', 'adopted_within_120_days')]:
            rate_dict = prior_stats.get(f'{col_key}_{target}_rate', {})
            default = gr[target]
            df[f'{col_key}_{short}_rate'] = df[col].map(rate_dict).fillna(default)

    # --- Comptages / métadonnées ---
    df['farmer_prior_count']  = df['farmer_name'].map(
        prior_stats.get('farmer_prior_count', {})).fillna(0)
    df['farmer_experienced']  = (df['farmer_prior_count'] > 1).astype(int)
    df['group_size']          = df['group_name'].map(
        prior_stats.get('group_size', {})).fillna(1)
    df['group_size_log']      = np.log1p(df['group_size'])
    df['farmer_prior_log']    = np.log1p(df['farmer_prior_count'])

    # --- Interactions clés ---
    for short in ['07', '90', '120']:
        df[f'coop_county_{short}'] = df['belong_to_cooperative'] * df[f'county_{short}_rate']
        df[f'gender_county_{short}'] = df['gender_enc'] * df[f'county_{short}_rate']
        df[f'topic_county_{short}'] = df[f'topic_avg_{short}_rate'] * df[f'county_{short}_rate']
        df[f'trainer_county_{short}'] = df[f'trainer_first_{short}_rate'] * df[f'county_{short}_rate']

    df['topics_coop']  = df['n_topics'] * df['belong_to_cooperative']
    df['age_coop']     = df['age_enc'] * df['belong_to_cooperative']
    df['experienced_coop'] = df['farmer_experienced'] * df['belong_to_cooperative']

    # --- Label encoding haute cardinalité ---
    for col in ['county', 'subcounty', 'ward', 'group_name', 'trainer_first']:
        le = LabelEncoder()
        df[f'{col}_le'] = le.fit_transform(df[col].fillna('unknown').astype(str))

    return df

def get_feature_cols(train_fe):
    """Récupère automatiquement toutes les colonnes numériques features."""
    exclude = [
        'ID', 'farmer_name', 'training_day', 'gender', 'age', 'registration',
        'group_name', 'county', 'subcounty', 'ward', 'trainer', 'topics_list',
        'trainer_first', 'topics_parsed', 'trainers_parsed',
        'adopted_within_07_days', 'adopted_within_90_days', 'adopted_within_120_days',
    ]
    cols = [c for c in train_fe.columns
            if c not in exclude
            and train_fe[c].dtype in [np.float64, np.int64, np.float32, np.int32]
            and not train_fe[c].isnull().all()]
    return cols


# ============================================================
# 5. MODÈLES
# ============================================================

def get_models(target):
    """Retourne la liste des modèles à entraîner."""
    # Poids de classe pour les données déséquilibrées
    pos_weights = {
        'adopted_within_07_days': 80,
        'adopted_within_90_days': 60,
        'adopted_within_120_days': 40,
    }
    pw = pos_weights[target]
    class_weight = {0: 1, 1: pw}

    models = {}

    # HistGradientBoosting (similaire à LightGBM, rapide)
    models['hgb'] = HistGradientBoostingClassifier(
        max_iter=500,
        learning_rate=0.05,
        max_depth=7,
        min_samples_leaf=20,
        l2_regularization=0.1,
        class_weight='balanced',
        early_stopping=True,
        validation_fraction=0.1,
        n_iter_no_change=30,
        random_state=42,
    )

    # Random Forest
    models['rf'] = RandomForestClassifier(
        n_estimators=500,
        max_depth=10,
        min_samples_leaf=10,
        class_weight=class_weight,
        n_jobs=-1,
        random_state=42,
    )

    if USE_LGBM:
        models['lgbm'] = lgb.LGBMClassifier(
            n_estimators=2000,
            learning_rate=0.03,
            num_leaves=63,
            max_depth=7,
            min_child_samples=30,
            subsample=0.8,
            colsample_bytree=0.8,
            scale_pos_weight=pw,
            reg_alpha=0.1,
            reg_lambda=1.0,
            random_state=42,
            n_jobs=-1,
            verbose=-1,
        )

    if USE_XGB:
        models['xgb'] = xgb.XGBClassifier(
            n_estimators=2000,
            learning_rate=0.03,
            max_depth=6,
            min_child_weight=10,
            subsample=0.8,
            colsample_bytree=0.8,
            scale_pos_weight=pw,
            random_state=42,
            n_jobs=-1,
            verbosity=0,
            early_stopping_rounds=100,
            eval_metric='logloss',
        )

    if USE_CATBOOST:
        models['catboost'] = CatBoostClassifier(
            iterations=2000,
            learning_rate=0.03,
            depth=7,
            scale_pos_weight=pw,
            random_seed=42,
            verbose=0,
            early_stopping_rounds=100,
        )

    return models


# ============================================================
# 6. CROSS-VALIDATION
# ============================================================

def train_cv(train_fe, test_fe, feature_cols, target, n_splits=5):
    """
    Entraîne tous les modèles en CV et retourne:
    - OOF predictions par modèle
    - Test predictions par modèle
    """
    X = train_fe[feature_cols].fillna(-1).values
    y = train_fe[target].values
    X_test = test_fe[feature_cols].fillna(-1).values

    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    models_dict = get_models(target)

    oof_preds  = {name: np.zeros(len(X))      for name in models_dict}
    test_preds = {name: np.zeros(len(X_test)) for name in models_dict}

    for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y)):
        print(f"    Fold {fold+1}/{n_splits}...", end=' ')
        X_tr, X_val = X[tr_idx], X[val_idx]
        y_tr, y_val = y[tr_idx], y[val_idx]

        for name, model in models_dict.items():
            m = model  # fresh copy not needed for sklearn since we refit

            # Gestion early stopping pour LightGBM et XGBoost
            if USE_LGBM and name == 'lgbm':
                m.fit(X_tr, y_tr,
                      eval_set=[(X_val, y_val)],
                      callbacks=[lgb.early_stopping(100, verbose=False),
                                 lgb.log_evaluation(-1)])
            elif USE_XGB and name == 'xgb':
                m.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
            elif USE_CATBOOST and name == 'catboost':
                m.fit(X_tr, y_tr, eval_set=(X_val, y_val), verbose=0)
            else:
                m.fit(X_tr, y_tr)

            oof_preds[name][val_idx] = m.predict_proba(X_val)[:, 1]
            test_preds[name] += m.predict_proba(X_test)[:, 1] / n_splits

        print("OK")

    return oof_preds, test_preds


# ============================================================
# 7. STACKING DE NIVEAU 2
# ============================================================

def stack(oof_preds, test_preds, y):
    """Stacking avec Logistic Regression calibrée."""
    model_names = list(oof_preds.keys())
    X_meta_train = np.column_stack([oof_preds[n] for n in model_names])
    X_meta_test  = np.column_stack([test_preds[n] for n in model_names])

    meta = LogisticRegression(C=0.1, max_iter=1000, random_state=42)
    meta.fit(X_meta_train, y)

    final_oof  = meta.predict_proba(X_meta_train)[:, 1]
    final_test = meta.predict_proba(X_meta_test)[:, 1]
    return final_oof, final_test


# ============================================================
# 8. ÉVALUATION
# ============================================================

def evaluate(y_true, y_pred, target_name):
    ll  = log_loss(y_true, y_pred)
    auc = roc_auc_score(y_true, y_pred)

    # F1 avec seuil optimisé sur données déséquilibrées
    thresholds = np.linspace(0.01, 0.5, 100)
    f1s = [f1_score(y_true, (y_pred >= t).astype(int), zero_division=0) for t in thresholds]
    best_f1  = max(f1s)
    best_thr = thresholds[np.argmax(f1s)]

    # Score challenge approximatif
    challenge_score = 0.75 * (1 - ll) + 0.25 * auc

    print(f"\n  ✓ [{target_name}]")
    print(f"    LogLoss:   {ll:.5f}")
    print(f"    ROC-AUC:   {auc:.5f}")
    print(f"    F1 (best): {best_f1:.5f} @ threshold={best_thr:.3f}")
    print(f"    Score≈:    {challenge_score:.5f}")
    return ll, auc, best_f1


# ============================================================
# 9. PIPELINE PRINCIPAL
# ============================================================

def main(data_path='./'):
    print("="*65)
    print("  DigiCow Adoption Prediction — Solution Haute Performance")
    print("="*65)

    # 1. Chargement
    print("\n[1] Chargement des données...")
    train, test, prior, sample = load_data(data_path)
    print(f"  Train: {train.shape} | Test: {test.shape} | Prior: {prior.shape}")
    print(f"  Taux adoption 7j:  {train['adopted_within_07_days'].mean():.3%}")
    print(f"  Taux adoption 90j: {train['adopted_within_90_days'].mean():.3%}")
    print(f"  Taux adoption 120j:{train['adopted_within_120_days'].mean():.3%}")

    # 2. Stats Prior
    print("\n[2] Construction des statistiques Prior...")
    prior_stats = build_prior_stats(prior)

    # 3. Feature Engineering
    print("\n[3] Feature Engineering...")
    train_fe = feature_engineer(train, prior_stats, is_train=True)
    test_fe  = feature_engineer(test,  prior_stats, is_train=False)
    feature_cols = get_feature_cols(train_fe)
    print(f"  Nombre de features: {len(feature_cols)}")

    # 4. Entraînement
    submission = sample.copy()
    results = {}

    for target, col_auc, col_ll in [
        ('adopted_within_07_days',  'Target_07_AUC', 'Target_07_LogLoss'),
        ('adopted_within_90_days',  'Target_90_AUC', 'Target_90_LogLoss'),
        ('adopted_within_120_days', 'Target_120_AUC', 'Target_120_LogLoss'),
    ]:
        print(f"\n[4] ══════ Entraînement: {target} ══════")
        y = train_fe[target].values

        # Cross-validation
        oof_preds, test_preds = train_cv(train_fe, test_fe, feature_cols, target, n_splits=5)

        # Stacking
        print("  Stacking niveau 2...")
        final_oof, final_test = stack(oof_preds, test_preds, y)

        # Évaluation
        ll, auc, f1 = evaluate(y, final_oof, target)
        results[target] = {'logloss': ll, 'auc': auc, 'f1': f1}

        # Remplissage submission
        # Note: les deux colonnes AUC et LogLoss reçoivent la même probabilité
        submission[col_auc] = final_test
        submission[col_ll]  = final_test

    # 5. Résumé final
    print("\n" + "="*65)
    print("  RÉSUMÉ FINAL (OOF Cross-Validation)")
    print("="*65)
    total_score = 0
    for target, metrics in results.items():
        score = 0.75 * (1 - metrics['logloss']) + 0.25 * metrics['auc']
        total_score += score / 3
        print(f"  {target[-3:]}j → LogLoss={metrics['logloss']:.4f} | "
              f"AUC={metrics['auc']:.4f} | F1={metrics['f1']:.4f} | Score≈{score:.4f}")
    print(f"\n  Score global estimé: {total_score:.5f}")

    # 6. Sauvegarde
    print("\n[5] Sauvegarde...")
    submission.to_csv('submission_final.csv', index=False)
    print("  ✓ submission_final.csv créé")

    # Diagnostic: distribution des prédictions
    print("\n  Distribution des prédictions test (adoption 7j):")
    preds_7 = submission['Target_07_AUC']
    print(f"    Min={preds_7.min():.4f} | Max={preds_7.max():.4f} | "
          f"Mean={preds_7.mean():.4f} | Median={preds_7.median():.4f}")

    return submission

if __name__ == '__main__':
    sub = main(data_path='./')
    print("\nTerminé ! Soumettez submission_v1.csv")

✓ LightGBM disponible
✓ XGBoost disponible
✓ CatBoost disponible
  DigiCow Adoption Prediction — Solution Haute Performance

[1] Chargement des données...
  Train: (13536, 17) | Test: (5621, 14) | Prior: (44882, 17)
  Taux adoption 7j:  1.130%
  Taux adoption 90j: 1.581%
  Taux adoption 120j:2.231%

[2] Construction des statistiques Prior...
  Calcul des stats Prior...

[3] Feature Engineering...
  Nombre de features: 87

[4] ══════ Entraînement: adopted_within_07_days ══════
    Fold 1/5... OK
    Fold 2/5... OK
    Fold 3/5... OK
    Fold 4/5... OK
    Fold 5/5... OK
  Stacking niveau 2...

  ✓ [adopted_within_07_days]
    LogLoss:   0.02776
    ROC-AUC:   0.98770
    F1 (best): 0.55705 @ threshold=0.356
    Score≈:    0.97610

[4] ══════ Entraînement: adopted_within_90_days ══════
    Fold 1/5... OK
    Fold 2/5... OK
    Fold 3/5... OK
    Fold 4/5... OK
    Fold 5/5... OK
  Stacking niveau 2...

  ✓ [adopted_within_90_days]
    LogLoss:   0.03670
    ROC-AUC:   0.97761
    F1 (bes